In [36]:
# If running in a fresh kernel:
!pip install -q google-genai textstat pandas numpy tqdm python-dotenv

In [37]:
from pathlib import Path
import re, json, time
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
import os

from google import genai
from google.genai import types

load_dotenv()
assert os.getenv("GEMINI_API_KEY"), "Set GEMINI_API_KEY in a .env file or environment."

# -------- Paths --------
DATA_DIR = Path("data_readability")          # <— put your 5 source .txt files here
OUT_DIR  = Path("outputs"); OUT_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR  = Path("results"); RES_DIR.mkdir(parents=True, exist_ok=True)

# -------- Experiment knobs --------
MODEL_NAME        = "gemini-2.5-pro"
GRADE_LEVELS      = [6, 5, 4, 3, 2]
TEMPERATURE       = 0.2
MAX_OUTPUT_TOKENS = 8096

# Reasoning length "auto": dynamic budget; if your SDK is older this is ignored gracefully.
THINKING_BUDGET   = -1    # set to None to disable

# “Hard word” definition (your request): ≥ 3 syllables
HARD_WORD_SYLLABLES = 3


In [38]:
def ordinal(n: int) -> str:
    return f"{n}{'th' if 11<=n%100<=13 else {1:'st',2:'nd',3:'rd'}.get(n%10,'th')}"

XML_PROMPT_TEMPLATE = """<?xml version="1.0" encoding="UTF-8"?>
<medical_writer_prompt>
  <role>Medical Writer (Patient Education)</role>
  <task>Translate complex medical information into clear, concise, and easily understandable plain text for a general audience.</task>
  <strict_instructions>
    <instruction>The final output MUST be plain text only.</instruction>
    <instruction>Do NOT use any special formatting (no Markdown, no HTML, no headers, no lists, no bolding).</instruction>
    <instruction>Directly output the final rewritten text. Omit all other words, including introductory phrases, conversational filler, or any explanations.</instruction>
  </strict_instructions>
  <guidelines>
    <guideline>
      <name>Target Reading Level</name>
      <value>{target_grade_text}</value>
    </guideline>
    <guideline>
      <name>Tone</name>
      <value>Professional, informative, respectful. Avoid condescension or childish language.</value>
    </guideline>
    <guideline>
      <name>Accuracy</name>
      <value>Maintain factual accuracy. Do not oversimplify to the point of losing important details or becoming misleading.</value>
    </guideline>
    <guideline>
      <name>Terminology</name>
      <rules>
        <rule>Replace complex medical terms with plain language equivalents.</rule>
        <rule>If a technical term is necessary, provide a brief, clear definition in parentheses immediately after.</rule>
        <rule>Use short, simple sentences and active voice.</rule>
        <rule>Avoid jargon, idioms, and metaphors.</rule>
      </rules>
    </guideline>
    <guideline>
      <name>Structure</name>
      <rules>
        <rule>Use short paragraphs (2-3 sentences).</rule>
      </rules>
    </guideline>
  </guidelines>
  <example>
    <input>
      Patients with hypertension should adhere to a DASH diet, which emphasizes consumption of fruits, vegetables, and low-fat dairy products, while minimizing intake of saturated fats, cholesterol, and trans fats.  This dietary modification has been shown to significantly reduce systolic and diastolic blood pressure.
    </input>
    <output>
If you have high blood pressure, also called hypertension, eating healthy can help manage it. A good eating plan is the DASH diet.

This plan means eating lots of fruits and vegetables. You should also choose low-fat dairy products like milk and yogurt. It is important to limit foods with unhealthy fats, such as fatty meats, butter, and many processed foods.

Following this diet can help lower your blood pressure and keep your heart healthy.
    </output>
  </example>
  <source>
    <![CDATA[
{source_text}
    ]]>
  </source>
</medical_writer_prompt>
"""



In [39]:
def load_materials_from_dir(base_dir: Path) -> pd.DataFrame:
    files = sorted(base_dir.glob("*.txt"))
    assert files, f"No .txt files found in {base_dir.resolve()}"
    rows = []
    for p in files:
        text = p.read_text(encoding="utf-8")
        stem = p.stem
        topic = re.sub(r"[_-]+", " ", stem).strip().title()
        rows.append({"id": stem, "topic": topic, "filename": p.name, "text": text})
    return pd.DataFrame(rows)

materials = load_materials_from_dir(DATA_DIR)
materials


,id,topic,filename,text
0,AAA,Aaa,AAA.txt,﻿A Guide for Patients:\nAbdominal Aortic\nAneu...
1,CAD,Cad,CAD.txt,﻿A Guide for Patients:\nCarotid Artery Disease...
2,DVT,Dvt,DVT.txt,﻿A Guide for Patients:\nDeep Vein Thrombosis\n...
3,PAD,Pad,PAD.txt,﻿A Guide for Patients: Peripheral Artery Disea...
4,VV,Vv,VV.txt,﻿A Guide for Patients: Varicose and Spider Vei...


In [40]:
import textstat

def _safe_div(a: float, b: float) -> float:
    return float(a)/float(b) if b else 0.0

WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def tokenize_words(text: str) -> List[str]:
    return WORD_RE.findall(text)

def hard_words_count(text: str, min_syllables: int = HARD_WORD_SYLLABLES) -> int:
    return sum(1 for w in tokenize_words(text) if textstat.syllable_count(w) >= min_syllables)

def dale_chall_grade_equiv(score: float) -> float:
    # Midpoints of DC bands (standard mapping)
    if score <= 4.9:  return 4.0
    if score <= 5.9:  return 5.5
    if score <= 6.9:  return 7.5
    if score <= 7.9:  return 9.5
    if score <= 8.9:  return 11.5
    return 14.0

def fry_components(text: str) -> Dict[str, float]:
    words = max(textstat.lexicon_count(text, removepunct=True), 1)
    sents = max(textstat.sentence_count(text), 1)
    syll  = max(textstat.syllable_count(text), 0)
    return {
        "fry_syllables_per_100": (syll / words) * 100.0,
        "fry_sentences_per_100": (sents / words) * 100.0
    }

def compute_all_metrics(text: str, include_forcast=True, include_spache=True, include_fry_components=True) -> Dict[str, Any]:
    wc  = textstat.lexicon_count(text, removepunct=True)
    sc  = textstat.sentence_count(text)
    syl = textstat.syllable_count(text)
    cc  = textstat.char_count(text, ignore_spaces=True)

    cpw = _safe_div(cc, wc)
    wps = _safe_div(wc, sc)
    spw = _safe_div(syl, wc)

    hard_ct  = hard_words_count(text, HARD_WORD_SYLLABLES)
    hard_pct = _safe_div(hard_ct, wc) * 100.0

    # Base formulas (grade-like)
    fkgl   = textstat.flesch_kincaid_grade(text)
    ari    = textstat.automated_readability_index(text)
    fog    = textstat.gunning_fog(text)
    cli    = textstat.coleman_liau_index(text)
    smog   = textstat.smog_index(text)
    lwrite = textstat.linsear_write_formula(text)
    dc_raw = textstat.dale_chall_readability_score(text)
    dc_geq = dale_chall_grade_equiv(dc_raw)

    # Pre-ARLC estimate (for gating)
    core = [fkgl, ari, fog, cli, smog, lwrite, dc_geq]
    pre_arlc = float(np.mean([x for x in core if x is not None and np.isfinite(x)])) if core else None

    # Optional/gated formulas
    forcast = None
    spache  = None
    if include_forcast and (pre_arlc is None or pre_arlc >= 7.0):   # FORCAST validated ≥7th grade
        # FORCAST grade = 20 − (N/10), N = monosyllabic words per 150 words
        words = tokenize_words(text)
        if words:
            mono = sum(1 for w in words if textstat.syllable_count(w) == 1)
            n_per_150 = (mono/len(words)) * 150.0
            forcast = 20.0 - (n_per_150 / 10.0)

    if include_spache and (pre_arlc is not None and pre_arlc <= 4.0): # Spache for <≈4th grade texts
        spache = textstat.spache_readability(text)

    # ARLC = mean of available grade-like formulas
    candidates = [fkgl, ari, fog, cli, smog, lwrite, dc_geq]
    if forcast is not None: candidates.append(forcast)
    if spache  is not None: candidates.append(spache)
    arlc = float(np.mean([x for x in candidates if x is not None and np.isfinite(x)])) if candidates else None

    metrics = {
        "word_count": wc,
        "sentence_count": sc,
        "syllables": syl,
        "characters_no_spaces": cc,
        "characters_per_word": cpw,
        "words_per_sentence": wps,
        "syllables_per_word": spw,
        "hard_words_count": hard_ct,
        "hard_words_percent": hard_pct,
        # core formulas
        "flesch_kincaid_grade": fkgl,
        "automated_readability_index": ari,
        "gunning_fog": fog,
        "coleman_liau": cli,
        "smog_index": smog,
        "linsear_write": lwrite,
        "dale_chall_raw": dc_raw,
        "dale_chall_grade_equiv": dc_geq,
        # gated
        "forcast_grade": forcast,
        "spache_grade": spache,
        # consensus
        "arlc": arlc,
    }
    if include_fry_components:
        metrics.update(fry_components(text))
    return metrics


In [41]:
client = genai.Client()

def _make_config():
    cfg_kwargs = dict(temperature=TEMPERATURE, max_output_tokens=MAX_OUTPUT_TOKENS)
    try:
        if THINKING_BUDGET is not None:
            cfg_kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    except Exception:
        pass
    return types.GenerateContentConfig(**cfg_kwargs)

def rewrite_with_gemini(source_text: str, target_grade: int) -> str:
    target_grade_text = f"{ordinal(target_grade)}-grade level"
    prompt = XML_PROMPT_TEMPLATE.format(
        target_grade_text=target_grade_text,
        source_text=source_text
    )
    cfg = _make_config()
    last_err = None
    for attempt in range(4):
        try:
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=cfg
            )
            return resp.text or ""
        except Exception as e:
            last_err = e
            time.sleep(1.2 * (attempt + 1))
    raise RuntimeError(f"Gemini call failed after retries: {last_err}")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [42]:
def run_study(materials_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    # Baseline metrics
    for _, r in materials_df.iterrows():
        orig_m = compute_all_metrics(r["text"])
        rows.append({
            "material_id": r["id"], "topic": r["topic"], "target_grade": "original",
            "model": "N/A", "output_path": "", "rewritten_snippet": "", **orig_m
        })

    # Rewrites
    for _, r in tqdm(materials_df.iterrows(), total=len(materials_df), desc="Rewriting"):
        out_dir = (OUT_DIR / r["id"]); out_dir.mkdir(parents=True, exist_ok=True)
        for g in GRADE_LEVELS:
            out_txt = out_dir / f"{MODEL_NAME}_g{g}.txt"
            if out_txt.exists():
                rewritten = out_txt.read_text(encoding="utf-8")
            else:
                rewritten = rewrite_with_gemini(r["text"], g)
                out_txt.write_text(rewritten, encoding="utf-8")

            m = compute_all_metrics(rewritten)
            rows.append({
                "material_id": r["id"], "topic": r["topic"], "target_grade": g,
                "model": MODEL_NAME, "output_path": str(out_txt),
                "rewritten_snippet": rewritten[:240].replace("\n", " "),
                **m
            })

    df = pd.DataFrame(rows)
    csv_path = RES_DIR / "metrics.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    (RES_DIR / "metrics.json").write_text(df.to_json(orient="records", indent=2), encoding="utf-8")
    print(f"Saved {csv_path} and results/metrics.json")
    return df

materials = load_materials_from_dir(DATA_DIR)
results = run_study(materials)
results.head()


Rewriting: 100%|██████████| 5/5 [08:57<00:00, 107.49s/it]

Saved results\metrics.csv and results/metrics.json


,material_id,topic,target_grade,model,output_path,rewritten_snippet,word_count,sentence_count,syllables,characters_no_spaces,...,coleman_liau,smog_index,linsear_write,dale_chall_raw,dale_chall_grade_equiv,forcast_grade,spache_grade,arlc,fry_syllables_per_100,fry_sentences_per_100
0,AAA,Aaa,original,N/A,,,540,25,919,2791,...,12.022963,15.381576,12.500000,11.433230,14.0,11.301115,None,13.509519,170.185185,4.629630
1,CAD,Cad,original,N/A,,,430,21,756,2357,...,13.359535,15.279682,12.166667,12.253352,14.0,11.789838,None,13.720243,175.813953,4.883721
2,DVT,Dvt,original,N/A,,,989,46,1748,5458,...,13.705763,15.763577,14.200000,12.685711,14.0,11.445663,None,14.329263,176.744186,4.651163
3,PAD,Pad,original,N/A,,,501,26,855,2654,...,12.323752,14.554593,8.857143,11.116266,14.0,11.185259,None,12.630636,170.658683,5.189621
4,VV,Vv,original,N/A,,,558,29,925,3022,...,13.085663,12.909473,7.000000,11.891625,14.0,11.720430,None,12.006748,165.770609,5.197133


In [43]:
# Mean ARLC by material × target grade
pivot = (results.query("target_grade != 'original'")
                  .pivot_table(values="arlc", index=["material_id","topic"],
                               columns="target_grade", aggfunc="mean"))
display(pivot)

# Compare original vs grade targets
summary_cols = ["material_id","topic","target_grade","arlc",
                "word_count","words_per_sentence","syllables_per_word","hard_words_percent"]
display(results[summary_cols].sort_values(["material_id","target_grade"]).head(20))


,target_grade,2,3,4,5,6
material_id,topic,,,,,
AAA,Aaa,5.842704,5.799418,6.309983,6.856120,7.799581
CAD,Cad,5.171462,5.895375,6.817147,7.794496,8.300465
DVT,Dvt,5.173457,6.151919,6.619997,7.300776,7.927394
PAD,Pad,4.971596,5.735860,6.443903,7.632521,8.184572
VV,Vv,4.679278,4.976805,5.481613,6.104028,6.762306


,material_id,topic,target_grade,arlc,word_count,words_per_sentence,syllables_per_word,hard_words_percent
9,AAA,Aaa,2,5.842704,416,11.885714,1.298077,6.250000
8,AAA,Aaa,3,5.799418,486,11.853659,1.283951,4.526749
7,AAA,Aaa,4,6.309983,471,12.076923,1.322718,6.157113
6,AAA,Aaa,5,6.856120,501,12.525000,1.341317,5.988024
5,AAA,Aaa,6,7.799581,477,12.891892,1.394130,9.643606
0,AAA,Aaa,original,13.509519,540,21.600000,1.701852,21.296296
14,CAD,Cad,2,5.171462,325,9.285714,1.252308,4.923077
13,CAD,Cad,3,5.895375,382,9.317073,1.345550,7.329843
12,CAD,Cad,4,6.817147,488,11.619048,1.327869,5.532787
11,CAD,Cad,5,7.794496,469,11.439024,1.411514,9.381663
